# `ToolRetryMiddleware`

Middleware that automatically retries failed tool calls using configurable exception filtering and backoff delays.

It can apply retry logic to every tool or only selected tools. When all retries are exhausted, it can return an error `ToolMessage`, re-raise the original exception, or generate custom error-message content.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
ToolRetryMiddleware(
    *,
    max_retries: int = 2, # Retries after the initial attempt
    tools: list[BaseTool | str] | None = None, # Tools to retry
    retry_on: RetryOn = (Exception,), # Retryable exceptions
    on_failure: OnFailure = "continue", # Final failure behaviour
    backoff_factor: float = 2.0, # Backoff multiplier
    initial_delay: float = 1.0, # Delay before the first retry
    max_delay: float = 60.0, # Maximum retry delay
    jitter: bool = True # Add ±25% random variation
)
```

## Parameters

* `max_retries` — Maximum number of retry attempts after the initial tool call.
  * Default: `2`
  * Must be greater than or equal to `0`.
  * Total possible attempts are `max_retries + 1`.

* `tools` — Optional tools to which retry logic should apply.
  * Default: `None`
  * `None` — Applies retry logic to all tools.
  * Empty list `[]` — Applies retry logic to no tools.
  * Strings are treated as tool names.
  * `BaseTool` objects are converted to their `.name` values.

* `retry_on` — Determines which exceptions trigger a retry.
  * Default: `(Exception,)`
  * May be a tuple of exception classes.
  * May be a callable receiving an exception and returning `True` or `False`.

* `on_failure` — Determines what happens when an exception is not retryable or all retries are exhausted.
  * `"continue"` — Returns an error `ToolMessage`.
  * `"error"` — Re-raises the original exception.
  * Callable — Receives the final exception and returns custom text for the `ToolMessage`.

* `backoff_factor` — Multiplier used for exponential backoff.
  * Default: `2.0`
  * Must be greater than or equal to `0`.
  * Set to `0.0` to use a constant delay.

* `initial_delay` — Delay in seconds before the first retry.
  * Default: `1.0`
  * Must be greater than or equal to `0`.

* `max_delay` — Maximum delay in seconds between retry attempts.
  * Default: `60.0`
  * Must be greater than or equal to `0`.

* `jitter` — Whether to add random variation of approximately `±25%` to retry delays.
  * Default: `True`
  * Helps prevent many failing calls from retrying simultaneously.

## Type Aliases

### `RetryOn`

Defines which exceptions are retryable.

```python
RetryOn = (
    tuple[type[Exception], ...]
    | Callable[[Exception], bool]
)
```

It may be:

* A tuple of exception classes checked using `isinstance`.
* A callable returning `True` when the exception should be retried.

### `OnFailure`

Defines how a final failure is handled.

```python
OnFailure = (
    Literal["error", "continue"]
    | Callable[[Exception], str]
)
```

Supported values:

* `"continue"` — Converts the failure into an error `ToolMessage`.
* `"error"` — Re-raises the exception.
* Callable — Produces custom `ToolMessage` content.

## Deprecated `on_failure` Values

The constructor accepts two deprecated aliases for backward compatibility.

### `"raise"`

```python
ToolRetryMiddleware(
    on_failure="raise"
)
```

It emits a `DeprecationWarning` and is converted to:

```python
on_failure="error"
```

### `"return_message"`

```python
ToolRetryMiddleware(
    on_failure="return_message"
)
```

It emits a `DeprecationWarning` and is converted to:

```python
on_failure="continue"
```

## Attributes

* `max_retries` — Number of retries allowed after the initial call.
* `_tool_filter` — Internal list of selected tool names, or `None` for all tools.
* `tools` — Empty list because this middleware does not register new agent tools.
* `retry_on` — Exception tuple or exception-filtering callable.
* `on_failure` — Final failure-handling strategy.
* `backoff_factor` — Exponential backoff multiplier.
* `initial_delay` — Delay before the first retry.
* `max_delay` — Maximum delay between retries.
* `jitter` — Whether retry-delay variation is enabled.

## Tool Filtering

The constructor converts configured tools into names:

```python
self._tool_filter = [
    tool.name if not isinstance(tool, str) else tool
    for tool in tools
]
```

Tool filtering behaves as follows:

| Configuration | Behaviour |
|---|---|
| `tools=None` | Retry every tool |
| `tools=[]` | Retry no tools |
| `tools=["search"]` | Retry only `search` |
| `tools=[search_tool]` | Retry the tool named by `search_tool.name` |
| Mixed strings and tools | Retry every selected name |

Tool names are not validated against the tools bound to the agent during middleware construction.

## Methods

### 1. `_should_retry_tool`

Checks whether retry logic applies to one tool.

```python
_should_retry_tool(
    self,
    tool_name: str
) -> bool
```

Returns `True` when:

```python
self._tool_filter is None
```

or when:

```python
tool_name in self._tool_filter
```

### 2. `_format_failure_message`

Creates the default error text after tool execution fails.

```python
@staticmethod
def _format_failure_message(
    tool_name: str,
    exc: Exception,
    attempts_made: int
) -> str
```

The output includes:

* Tool name.
* Number of attempts.
* Exception class name.
* Exception message.
* A request to try again.

Example:

```text
Tool 'search' failed after 3 attempts with TimeoutError:
Request timed out. Please try again.
```

The method uses the singular word `"attempt"` when `attempts_made == 1`.

### 3. `_handle_failure`

Applies the configured final failure behaviour.

```python
_handle_failure(
    self,
    tool_name: str,
    tool_call_id: str | None,
    exc: Exception,
    attempts_made: int
) -> ToolMessage
```

#### Behaviour

When:

```python
on_failure="error"
```

the original exception is re-raised.

When `on_failure` is callable:

```python
content = self.on_failure(exc)
```

Otherwise, `_format_failure_message` is used.

For non-raising strategies, the method returns:

```python
ToolMessage(
    content=content,
    tool_call_id=tool_call_id,
    name=tool_name,
    status="error",
)
```

### 4. `wrap_tool_call`

Executes a synchronous tool call with retry logic.

```python
wrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        ToolMessage | Command[Any]
    ]
) -> ToolMessage | Command[Any]
```

#### Behaviour

1. Resolves the tool name.
2. Checks whether the tool is selected for retry handling.
3. Calls the handler normally when the tool is not selected.
4. Makes the initial attempt.
5. Catches standard exceptions.
6. Checks whether each exception is retryable.
7. Waits using `time.sleep` before another attempt.
8. Returns immediately after a successful result.
9. Applies `on_failure` when no retry remains.

The tool name is resolved using:

```python
tool_name = (
    request.tool.name
    if request.tool
    else request.tool_call["name"]
)
```

The tool-call ID is read from:

```python
request.tool_call["id"]
```

### 5. `awrap_tool_call`

Asynchronous version of `wrap_tool_call`.

```python
async def awrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        Awaitable[
            ToolMessage | Command[Any]
        ]
    ]
) -> ToolMessage | Command[Any]
```

It applies the same selection, retry, and failure-handling logic.

Differences from the synchronous method:

* Awaits the tool handler.
* Uses `asyncio.sleep` for retry delays.

## Retry Flow

```text
Receive ToolCallRequest
        |
        v
Should retry logic apply to this tool?
        |
   No --+--> Execute the handler once
        |
       Yes
        |
        v
Execute initial attempt
        |
        v
Did it succeed? -------------- Yes -------> Return ToolMessage or Command
        |
        No
        |
        v
Is this GraphBubbleUp? -------- Yes -------> Re-raise immediately
        |
        No
        |
        v
Is the exception retryable? --- No --------> Apply on_failure
        |
       Yes
        |
        v
Are retries remaining? ------- No --------> Apply on_failure
        |
       Yes
        |
        v
Calculate delay -> Wait -> Call the handler again
```

## `GraphBubbleUp` Handling

`GraphBubbleUp` is always re-raised immediately.

```python
except GraphBubbleUp:
    raise
```

It is not:

* Evaluated by `retry_on`.
* Retried.
* Converted into an error `ToolMessage`.
* Passed to a custom `on_failure` callable.

This preserves LangGraph control-flow signals such as interrupts and parent commands.

## Retry Attempts

The retry loop runs:

```python
for attempt in range(
    self.max_retries + 1
):
```

Therefore:

```text
max_retries = 0 -> 1 total attempt
max_retries = 1 -> 2 total attempts
max_retries = 2 -> 3 total attempts
```

The number passed to failure formatting is:

```python
attempts_made = attempt + 1
```

## Retryable Exceptions

### Exception Tuple

```python
retry = ToolRetryMiddleware(
    retry_on=(
        TimeoutError,
        ConnectionError,
    )
)
```

Only matching exception types are retried.

A non-matching exception is handled immediately according to `on_failure`, even when retry attempts remain.

### Callable Filter

```python
def should_retry(
    exc: Exception
) -> bool:
    return isinstance(
        exc,
        TimeoutError
    )

retry = ToolRetryMiddleware(
    retry_on=should_retry
)
```

The callable runs after every caught standard exception.

## Backoff Calculation

The delay is calculated by the shared `calculate_delay` helper.

Without constant-backoff mode, the base pattern is:

```python
delay = initial_delay * (
    backoff_factor ** retry_number
)
```

where `retry_number` is zero-based.

The result is capped at `max_delay`.

### Example

```python
ToolRetryMiddleware(
    max_retries=4,
    initial_delay=1.0,
    backoff_factor=2.0,
    max_delay=60.0,
    jitter=False,
)
```

Delays:

```text
Before retry 1: 1 × 2⁰ = 1 second
Before retry 2: 1 × 2¹ = 2 seconds
Before retry 3: 1 × 2² = 4 seconds
Before retry 4: 1 × 2³ = 8 seconds
```

### Constant Delay

Set `backoff_factor=0.0`:

```python
ToolRetryMiddleware(
    max_retries=3,
    initial_delay=2.0,
    backoff_factor=0.0,
    jitter=False,
)
```

Delays:

```text
Before retry 1: 2 seconds
Before retry 2: 2 seconds
Before retry 3: 2 seconds
```

### Disable Waiting

```python
ToolRetryMiddleware(
    max_retries=3,
    initial_delay=0.0,
)
```

The retry loop continues immediately because sleeping occurs only when:

```python
delay > 0
```

### Jitter

When `jitter=True`, random variation of approximately `±25%` is added to the calculated delay.

For a calculated delay of `4` seconds, the actual wait is approximately:

```text
3 to 5 seconds
```

## Final Failure Handling

### Continue Agent Execution

```python
retry = ToolRetryMiddleware(
    max_retries=2,
    on_failure="continue",
)
```

After the final failed attempt, the middleware returns an error `ToolMessage`.

Example content:

```text
Tool 'search' failed after 3 attempts with TimeoutError:
Request timed out. Please try again.
```

This allows the model to inspect the tool error and potentially recover.

### Re-raise the Exception

```python
retry = ToolRetryMiddleware(
    max_retries=2,
    on_failure="error",
)
```

The original exception is raised after the final failed attempt or immediately for a non-retryable exception.

### Custom Error Text

```python
def format_error(
    exc: Exception
) -> str:
    return (
        "The database is temporarily unavailable. "
        "Try a different approach."
    )

retry = ToolRetryMiddleware(
    tools=["search_database"],
    on_failure=format_error,
)
```

The callable's returned string becomes the `ToolMessage.content`.

The returned message still has:

```python
status="error"
```

## Return Types

A successful real tool handler may return:

```python
ToolMessage | Command[Any]
```

The middleware returns that successful result unchanged.

A handled final failure always returns:

```python
ToolMessage
```

unless:

```python
on_failure="error"
```

causes the exception to be raised.

## Validation

The constructor calls:

```python
validate_retry_params(
    max_retries,
    initial_delay,
    max_delay,
    backoff_factor,
)
```

It raises `ValueError` when:

```text
max_retries < 0
initial_delay < 0
max_delay < 0
backoff_factor < 0
```

## Defensive `RuntimeError`

Both retry methods contain an unreachable fallback:

```python
raise RuntimeError(
    "Unexpected: retry loop completed without returning"
)
```

Normal execution always returns through:

* A successful handler call.
* `_handle_failure`.

## Examples

### Basic Usage

```python
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolRetryMiddleware,
)

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[search_tool],
    middleware=[
        ToolRetryMiddleware()
    ],
)
```

This permits two retries after the initial tool call.

### Retry Selected Tools

```python
retry = ToolRetryMiddleware(
    tools=[
        "search_database",
        "fetch_remote_file",
    ],
    max_retries=4,
)
```

Other tools execute once without retry interception.

### Select Tools Using `BaseTool` Objects

```python
retry = ToolRetryMiddleware(
    tools=[
        search_database,
        fetch_remote_file,
    ],
    max_retries=4,
)
```

Their `.name` values are stored internally.

### Retry Specific Exceptions

```python
from requests.exceptions import (
    RequestException,
    Timeout,
)

retry = ToolRetryMiddleware(
    max_retries=4,
    retry_on=(
        RequestException,
        Timeout,
    ),
    backoff_factor=1.5,
)
```

### Retry Server Errors Only

```python
from requests.exceptions import HTTPError

def should_retry(
    exc: Exception
) -> bool:
    if isinstance(
        exc,
        HTTPError,
    ):
        response = exc.response
        return (
            response is not None
            and 500 <= response.status_code < 600
        )

    return False

retry = ToolRetryMiddleware(
    max_retries=3,
    retry_on=should_retry,
)
```

### Custom Failure Message

```python
def format_error(
    exc: Exception
) -> str:
    return (
        "Database temporarily unavailable. "
        "Please try again later."
    )

retry = ToolRetryMiddleware(
    max_retries=4,
    tools=["search_database"],
    on_failure=format_error,
)
```

### Constant Backoff

```python
retry = ToolRetryMiddleware(
    max_retries=5,
    backoff_factor=0.0,
    initial_delay=2.0,
    jitter=False,
)
```

### Raise After Failure

```python
retry = ToolRetryMiddleware(
    max_retries=2,
    on_failure="error",
)
```

### No Retry for Any Tool

```python
retry = ToolRetryMiddleware(
    tools=[],
    max_retries=3,
)
```

Every call is delegated directly to its handler once.

## Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/tool_retry.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```